# Gemma 3 (1B) Fine-Tune — MEB Soru Üretici

MEB müfredatına uygun Türkçe sorular üreten bir model eğitiyorum. Temel model olarak
`unsloth/gemma-3-1b-it` kullanıyorum, LoRA ile eğitip adaptörü Hugging Face'e yüklüyorum.

Notlar:
- Runtime → GPU (A100 / L4)
- Veri dosyası (`meb_identity_format_temiz.jsonl`) Drive'da
- `HF_TOKEN` Colab Secrets'ta (write yetkili)
- Gemma erişimini bir kez onaylamak gerekiyor

## 1) Kurulum

In [ ]:
!pip install -q unsloth
!pip install -q --force-reinstall --no-deps git+https://github.com/unslothai/unsloth.git

## 2) Hugging Face girişi

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 3) Modeli yükle

In [ ]:
from unsloth import FastModel
import torch

MODEL_ID = "unsloth/gemma-3-1b-it"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    full_finetuning = False,
)
print("Model yuklendi:", MODEL_ID)

## 4) LoRA ekle

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA eklendi.")

## 5) Veriyi Drive'dan yükle

`DATA_PATH` kendi Drive konumuma göre.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json

DATA_PATH = "/content/drive/MyDrive/meb_identity_format_temiz.jsonl"
assert os.path.exists(DATA_PATH), f"Dosya bulunamadi: {DATA_PATH} — yolu kontrol et"

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Toplam kayit:", len(records))
print("Ornek user :", records[0][0]["content"])
print("Ornek asst :", records[0][1]["content"][:120])

## 6) Sohbet formatına çevir

Her kaydı user/assistant mesajına çevirip Gemma sohbet şablonunu uyguluyorum.

In [ ]:
from datasets import Dataset

def to_messages(rec):
    return {"messages": [
        {"role": "user", "content": rec[0]["content"]},
        {"role": "assistant", "content": rec[1]["content"]},
    ]}

data = [to_messages(r) for r in records]
dataset = Dataset.from_list(data)

def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_template)
print(dataset)
print("\n--- Bir ornegin sablonlanmis hali ---")
print(dataset[0]["text"][:400])

## 7) Eğitim

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps = 20,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        max_length = MAX_SEQ_LENGTH,
    ),
)

train_on_responses_only ile sadece cevap kısmı üzerinden eğitiyorum.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
trainer_stats = trainer.train()

## 8) Modeli dene

In [ ]:
from transformers import TextStreamer

def uret(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.8, top_p=0.9, do_sample=True,
        repetition_penalty=1.3, no_repeat_ngram_size=3,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print(uret("8. sınıf Türkçe - Fiilimsiler konusunda orta seviyede çoktan seçmeli soru üret."))
print("\n" + "="*60 + "\n")
print(uret("5. sınıf Fen Bilimleri - Güneş, Dünya ve Ay konusunda orta seviyede çoktan seçmeli soru üret."))

## 9) LoRA adaptörünü Hugging Face'e yükle

In [ ]:
HF_USERNAME = "nursimakgul"
REPO_NAME = "gemma-3-1b-meb-soru-lora"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

SAVE_DIR = "gemma-3-1b-meb-lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Kaydedildi:", SAVE_DIR)

In [ ]:
readme_content = f"""---
base_model: {MODEL_ID}
library_name: peft
license: gemma
language:
- tr
tags:
- gemma
- gemma-3
- unsloth
- lora
- peft
- turkish
- education
- meb
- soru-uretimi
pipeline_tag: text-generation
---

# {REPO_NAME}

`{MODEL_ID}` temel alinarak, MEB mufredatina uygun Turkce egitim sorulari ureten bir
LoRA adaptoru. Unsloth ile fine-tune edilmistir.

## Amac

Kullaniciya sinif, ders ve konu verildiginde o konuya uygun coktan secmeli / acik uclu /
beceri temelli sorular uretmek. Ornek istem:

> "8. sinif Turkce - Fiilimsiler konusunda orta seviyede coktan secmeli soru uret."

Model, cevabi yapilandirilmis (JSON) bir soru formatinda uretir: soru metni, secenekler,
dogru cevap ve cevap aciklamasi.

## Egitim detaylari

- **Temel model:** {MODEL_ID}
- **Framework:** Unsloth + TRL (SFT)
- **Yontem:** LoRA (r=16, alpha=16), 4-bit (QLoRA)
- **Hedef moduller:** q/k/v/o_proj, gate/up/down_proj
- **Epoch:** 2
- **Veri:** MEB soru arsivinden turetilmis, referans sohbet formatinda ~20.000 ornek
  (matematik haric)

## Kullanim

```python
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained("{REPO_ID}", load_in_4bit=True)

messages = [{{"role": "user", "content": "5. sinif Fen Bilimleri - Gunes, Dunya ve Ay konusunda orta seviyede coktan secmeli soru uret."}}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.8, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
```

## Sinirlamalar

Uretilen sorularin dogrulugu her zaman garanti degildir; egitim/degerlendirme amaclidir,
dogrudan sinavda kullanim icin ek kontrol gerekir.
"""

with open(f"{SAVE_DIR}/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)
print("README yazildi.")

In [ ]:
from huggingface_hub import create_repo, upload_folder

create_repo(REPO_ID, repo_type="model", exist_ok=True)
upload_folder(folder_path=SAVE_DIR, repo_id=REPO_ID, repo_type="model")
print(f"\nTamamlandi! LoRA adaptoru: https://huggingface.co/{REPO_ID}")